In [ ]:
import pandas as pd
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()

In [53]:
import numpy as np

In [56]:
ruta = rc.validar_ruta()
now= pd.to_datetime('now')
cur_year = now.year 


In [ ]:

ruta_historoica = ruta / 'file' / f'ventas_historicas.xlsx'
ventas_historicas=pd.read_excel(ruta_historoica)

def limpiar_documento(valor):
    valor = str(valor).strip()
    valor = valor.replace('.', '').replace("'",'').replace("”", '').replace("’",'')  # Elimina espacios en blanco

    # Si el valor es NaN o vacío, retorna NaN
    if pd.isna(valor) or str(valor).strip() == pd.NaT:
        return pd.NA
    # Si el valor contiene solo letras, retorna NaN
    if re.fullmatch(r'[A-Za-z]+', str(valor).strip()):
        return pd.NA
    # Extrae los dígitos iniciales antes de cualquier carácter no numérico
    match = re.match(r'^(\d+)', str(valor))
    if match:
        return match.group(1)
    return pd.NA

ventas_historicas['IDENTIFICACION_CLIENTE'] = ventas_historicas['IDENTIFICACION_CLIENTE'].apply(limpiar_documento)
ventas_historicas['FECHA_FACTURA'] = pd.to_datetime(ventas_historicas['FECHA_FACTURA'])

ventas_historicas_agru = ventas_historicas.groupby('IDENTIFICACION_CLIENTE')['FECHA_FACTURA'].min().reset_index()

In [44]:
ventas_historicas_agru.to_excel(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\file\ventas_historicas.xlsx", index=False)

In [23]:
curren_sale = pd.read_excel(ruta / 'file' / f'BASE VENTAS {cur_year}.xlsx')

In [ ]:

# porcesar base de ventas y notas credito
ventas_procesadas = rc.transformar_base(origen=True)
ruta = rc.validar_ruta()

ruta_clean = ruta / 'CLEAN DATA' 
ruta2 = Path(ventas_procesadas['nombre_archivo'])
ruta_carpeta = ruta_clean / f'VENTAS_{ruta2.stem}.xlsx'
ruta_errores = ruta / 'file' / 'ventas_sin_categoria.xlsx'








In [47]:
ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'] = ventas_procesadas['Base']['IDENTIFICACION_CLIENTE'].apply(limpiar_documento)

In [48]:
df_resultado = ventas_procesadas['Base']

In [ ]:
ventas_historicas_agru

In [51]:
df_resultado = df_resultado.merge(ventas_historicas_agru, on='IDENTIFICACION_CLIENTE', how='left', suffixes=('', '_FECHA_MIN'))

In [79]:
df_resultado['CLIENTES NUEVOS'] = np.where(
    ((df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.month == now.month) & 
    (df_resultado['FECHA_FACTURA_FECHA_MIN'].dt.year == now.year))|(df_resultado['FECHA_FACTURA_FECHA_MIN'].isna()),
    "Cliente sin historico",
    "No es cliente nuevo"
)


In [78]:
df_resultado.to_excel(r"C:\Users\Dataa\Desktop\verificacion.xlsx")

In [69]:
df_resultado['CLIENTES NUEVOS'].value_counts()

CLIENTES NUEVOS
    23500
Name: count, dtype: int64

In [ ]:

try:
    ruta_clean.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_clean}' creada o ya existe.")
    ventas_procesadas['Base'].to_excel(ruta_carpeta, index=False)
    ventas_procesadas['errores'].to_excel(ruta_errores, index=False)
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")
# Consolidar ventas
base_clean = rc.consolidar_carpeta(ruta_carpeta=ruta_clean)
ruta_base = ruta / 'file' / 'BASE VENTAS 2025.xlsx'
import locale
try:
    # Intentamos usar el locale en español para obtener "ENERO", "FEBRERO", etc.
    locale.setlocale(locale.LC_TIME, 'es_ES.UTF-8')
except locale.Error:
    print("   - Advertencia: Locale 'es_ES.UTF-8' no disponible. Se usarán nombres de mes en inglés.")
    

    

base_clean['MES'] = base_clean['FECHA_FACTURA'].dt.strftime('%B').str.upper()
columnas_finales = [
        "Source.Name", "NUMERO_FACTURA", "FECHA_FACTURA", "AÑO", "MES", "DIA",
        "CLIENTE", "IDENTIFICACION_CLIENTE", "CATEGORÍA", "PRODUCTO", "CANTIDAD",
        "TOTAL", "TASA_CAMBIO", "TRM", "TOTAL($)", "TELEFONO", "EMAIL", "PAIS",
        "CIUDAD", "CIUDAD_CORREGIDA", "DEPARTAMENTO", "EQUIPO_VENTAS", "REFERENCIA"
    ]
    
# Manejo defensivo por si la columna 'ASESOR COMERCIAL' no siempre existe
if 'ASESOR COMERCIAL' in base_clean.columns:
    base_clean['ASESOR COMERCIAL'] = base_clean['ASESOR COMERCIAL'].astype(str)
    columnas_finales.append('ASESOR COMERCIAL')

# Aseguramos que solo reordenamos las columnas que realmente existen en el DataFrame
columnas_existentes = [col for col in columnas_finales if col in base_clean.columns]
base_clean = base_clean[columnas_existentes]

# Esta linea mantiene solo los pruductos comerciales
base_clean = base_clean[base_clean['PRODUCTO'].str.startswith(('[PCN','[KD','[TNG','[B8'))]   ###### linea modificada
try:
    ruta_file = ruta / 'file' 
    ruta_file.mkdir(parents=True, exist_ok=True)  # Crear la carpeta si no existe
    print(f"Carpeta '{ruta_file}' creada o ya existe.")
    base_clean.to_excel(ruta_base, index=False)
    
except Exception as e:
    print(f"Error al crear la carpeta o guardar los archivos: {e}")


explosion = rc.explosion_ventas()




In [ ]:
ventas_anio_pre.shape

In [ ]:
# Limpia la columna DOCUMENTO: elimina filas con solo texto y extrae solo los dígitos iniciales

def limpiar_documento(valor):
    valor = str(valor).strip()
    valor = valor.replace('.', '').replace("'",'').replace("”", '').replace("’",'')  # Elimina espacios en blanco

    # Si el valor es NaN o vacío, retorna NaN
    if pd.isna(valor) or str(valor).strip() == pd.NaT:
        return pd.NA
    # Si el valor contiene solo letras, retorna NaN
    if re.fullmatch(r'[A-Za-z]+', str(valor).strip()):
        return pd.NA
    # Extrae los dígitos iniciales antes de cualquier carácter no numérico
    match = re.match(r'^(\d+)', str(valor))
    if match:
        return match.group(1)
    return pd.NA

ventas_anio_pre['DOCUMENTO'] = ventas_anio_pre['DOCUMENTO'].apply(limpiar_documento)


In [ ]:

ventas_anio_pre = ventas_anio_pre.dropna(subset=['DOCUMENTO'])

In [ ]:
ventas_anio_pre.columns

In [ ]:
ventas_anio_pre['Fecha'] = pd.to_datetime(ventas_anio_pre['Fecha'], errors='coerce')

In [ ]:
ventas_anio_pre = ventas_anio_pre.groupby('DOCUMENTO')['Fecha'].min().reset_index()

In [ ]:
ventas_anio_pre.to_excel(r"C:\Users\Dataa\Desktop\verificacion.xlsx")

In [ ]:
ventas_anio_pre[ventas_anio_pre['DOCUMENTO'].str.contains(r'^\D+$', na=False)]['DOCUMENTO'].to_excel(r"C:\Users\Dataa\Desktop\verificacion.xlsx")

In [ ]:
cur_sales= pd.read_excel(ruta_year)

In [ ]:
ventas_anio_pre['DOCUMENTO'].replace(, '', regex=True)